# 07 · Policy validation v1

Evaluates the pre-frozen marginal-conformal policy family on the reserved policy-validation role; no tuning.

**Checkpoint date:** 2026-10-05.

In [ ]:
#!/usr/bin/env python3
"""
Evaluate the frozen 72-hour Gray-Box operational policy on the reserved
policy_validation role.

This is the FIRST stage allowed to read policy_validation outcomes.

Frozen before this script:
- probability calibration: raw/raw/raw
- conformal thresholds: primary alpha=0.10
- candidate policy family: q80/q90/q95
- primary policy: q90
- binary alarm thresholds for SHARP/AIA/fusion

No threshold or policy tuning is performed here.
No later roles (Cycle 25 / supplementary 2026) are evaluated here.
No model fitting/retraining.
No GPU use.
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math
import numpy as np
import pandas as pd

MASTER = Path("/home/abmoses2000/graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz")
MASTER_SHA = "17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"

SHARP = Path("/home/abmoses2000/sharp72_predictions_frozen_20261002.csv.gz")
SHARP_SHA = "7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"

CONF_DIR = Path("/home/abmoses2000/graybox_aia72_conformal_v1_20261005")
CONF_PROTOCOL = CONF_DIR / "protocol.json"
CONF_THRESHOLDS = CONF_DIR / "thresholds.csv"

POLICY_DIR = Path("/home/abmoses2000/graybox_aia72_policy_freeze_v1_20261005")
POLICY_PROTOCOL = POLICY_DIR / "protocol.json"
POLICY_FAMILY = POLICY_DIR / "policy_family.json"
CUTOFFS = POLICY_DIR / "candidate_cutoffs.csv"

ALARM_DIR = Path("/home/abmoses2000/graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005")
ALARM_PROTOCOL = ALARM_DIR / "protocol.json"
ALARM_THRESHOLDS = ALARM_DIR / "alarm_thresholds.csv"
PRIMARY_DECL = ALARM_DIR / "primary_policy_declaration.json"

OUT = Path("/home/abmoses2000/graybox_aia72_policy_validation_v1_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

ROLE = "policy_validation"
EXPECTED_ROWS = 13142
PRIMARY_ALPHA = 0.10
SEEDS = [17,29,43]

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond, msg):
    if not cond:
        raise ValueError(msg)

def entropy_binary(p):
    p = np.asarray(p, dtype=float)
    q = np.clip(p, 1e-12, 1-1e-12)
    return -(q*np.log(q) + (1-q)*np.log(1-q))

def conformal_state(p, qhat):
    p = np.asarray(p, dtype=float)
    include0 = p <= qhat
    include1 = (1.0-p) <= qhat
    out = np.full(len(p), "unavailable", dtype=object)
    finite = np.isfinite(p)
    out[finite & (~include0) & (~include1)] = "empty"
    out[finite & include0 & (~include1)] = "singleton_no_flare"
    out[finite & (~include0) & include1] = "singleton_flare"
    out[finite & include0 & include1] = "doubleton"
    return out

def confusion(y, pred):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    tp = int(((y==1)&(pred==1)).sum())
    fp = int(((y==0)&(pred==1)).sum())
    fn = int(((y==1)&(pred==0)).sum())
    tn = int(((y==0)&(pred==0)).sum())
    return tp,fp,fn,tn

def binary_metrics(y, pred):
    tp,fp,fn,tn = confusion(y,pred)
    recall = tp/(tp+fn) if (tp+fn) else np.nan
    specificity = tn/(tn+fp) if (tn+fp) else np.nan
    precision = tp/(tp+fp) if (tp+fp) else np.nan
    f1 = 2*precision*recall/(precision+recall) if np.isfinite(precision) and np.isfinite(recall) and (precision+recall)>0 else np.nan
    tss = recall + specificity - 1 if np.isfinite(recall) and np.isfinite(specificity) else np.nan
    denom = ((tp+fn)*(fn+tn)+(tp+fp)*(fp+tn))
    hss = 2*(tp*tn-fn*fp)/denom if denom else np.nan
    return {
        "tp":tp,"fp":fp,"fn":fn,"tn":tn,
        "recall":float(recall) if np.isfinite(recall) else None,
        "specificity":float(specificity) if np.isfinite(specificity) else None,
        "precision":float(precision) if np.isfinite(precision) else None,
        "f1":float(f1) if np.isfinite(f1) else None,
        "tss":float(tss) if np.isfinite(tss) else None,
        "hss":float(hss) if np.isfinite(hss) else None,
    }

def brier(y,p):
    mask = np.isfinite(p)
    return float(np.mean((p[mask]-y[mask])**2)) if mask.any() else None

def logloss(y,p):
    mask = np.isfinite(p)
    if not mask.any():
        return None
    q=np.clip(p[mask],1e-12,1-1e-12)
    yy=y[mask]
    return float(-np.mean(yy*np.log(q)+(1-yy)*np.log(1-q)))

def conformal_contains_true(y,p,qhat):
    finite=np.isfinite(p)
    include0 = finite & (p <= qhat)
    include1 = finite & ((1-p) <= qhat)
    contains = np.zeros(len(p),dtype=bool)
    contains[(y==0)] = include0[(y==0)]
    contains[(y==1)] = include1[(y==1)]
    size = include0.astype(int)+include1.astype(int)
    return finite, contains, size

# ---------- Verify frozen inputs ----------
for p in [MASTER,SHARP,CONF_PROTOCOL,CONF_THRESHOLDS,POLICY_PROTOCOL,POLICY_FAMILY,CUTOFFS,ALARM_PROTOCOL,ALARM_THRESHOLDS,PRIMARY_DECL]:
    require(p.is_file(), f"Missing frozen input: {p}")
require(sha256(MASTER)==MASTER_SHA, "Master SHA changed")
require(sha256(SHARP)==SHARP_SHA, "SHARP SHA changed")

conf_protocol=json.loads(CONF_PROTOCOL.read_text())
policy_protocol=json.loads(POLICY_PROTOCOL.read_text())
policy_family=json.loads(POLICY_FAMILY.read_text())
alarm_protocol=json.loads(ALARM_PROTOCOL.read_text())
primary_decl=json.loads(PRIMARY_DECL.read_text())

require(conf_protocol["primary_alpha"]==PRIMARY_ALPHA, "Primary conformal alpha changed")
require(policy_protocol["policy_validation_outcomes_read"] is False, "Policy freeze had already read outcomes")
require(alarm_protocol["policy_validation_outcomes_read"] is False, "Alarm freeze had already read outcomes")
require(primary_decl["primary_policy_id"]=="policy_q90", "Primary policy declaration changed")
require(primary_decl["selection_on_policy_validation_allowed"] is False, "Primary policy may not be selected on validation")

cutoffs=pd.read_csv(CUTOFFS)
conf_thresholds=pd.read_csv(CONF_THRESHOLDS)
alarm_thresholds=pd.read_csv(ALARM_THRESHOLDS)

alarm_map={r.branch:float(r.threshold) for r in alarm_thresholds.itertuples()}
require(set(alarm_map)=={"sharp","aia","sharp_aia"}, "Alarm threshold branches changed")

qhat={}
for branch in ["sharp","aia","sharp_aia"]:
    hit=conf_thresholds.loc[(conf_thresholds["branch"]==branch)&np.isclose(conf_thresholds["alpha"].astype(float),PRIMARY_ALPHA)]
    require(len(hit)==1, f"Missing primary qhat: {branch}")
    qhat[branch]=float(hit.iloc[0]["qhat_nonconformity"])

# ---------- Open policy_validation role ----------
master=pd.read_csv(MASTER,low_memory=False)
sharp=pd.read_csv(SHARP,low_memory=False)

a=master.loc[master["role"].astype(str)==ROLE].copy()
s=sharp.loc[sharp["role"].astype(str)==ROLE].copy()
require(len(a)==EXPECTED_ROWS and len(s)==EXPECTED_ROWS, "Policy-validation support changed")

a["forecast_case_id"]=a["forecast_case_id"].astype(str)
s["forecast_case_id"]=s["forecast_case_id"].astype(str)
require(a["forecast_case_id"].tolist()==s["forecast_case_id"].tolist(), "SHARP/AIA policy-validation IDs/order mismatch")

y=pd.to_numeric(a["label"],errors="raise").astype(int).to_numpy()
ys=pd.to_numeric(s["label"],errors="raise").astype(int).to_numpy()
require(np.array_equal(y,ys), "SHARP/AIA policy-validation labels disagree")
require(set(np.unique(y)).issubset({0,1}), "Non-binary labels")

aia_seed=a[[f"probability_seed_{z}" for z in SEEDS]].astype(float).to_numpy()
sharp_seed=s[[f"probability_seed_{z}" for z in SEEDS]].astype(float).to_numpy()
aia_p=pd.to_numeric(a["probability"],errors="coerce").to_numpy(float)
sharp_p=pd.to_numeric(s["probability_gru_mean"],errors="coerce").to_numpy(float)
fusion_seed=0.5*sharp_seed+0.5*aia_seed
fusion_p=0.5*sharp_p+0.5*aia_p

# Availability and status.
aia_status=a["input_status"].astype(str).to_numpy() if "input_status" in a.columns else np.array(["ok"]*len(a),dtype=object)
aia_ok=(aia_status=="ok") & np.isfinite(aia_p) & np.isfinite(aia_seed).all(axis=1)
sharp_ok=np.isfinite(sharp_p) & np.isfinite(sharp_seed).all(axis=1)
fusion_ok=aia_ok & sharp_ok & np.isfinite(fusion_p)

# Signals.
sig=pd.DataFrame({
    "forecast_case_id":a["forecast_case_id"].to_numpy(),
    "issue_utc":a["issue_utc"].to_numpy() if "issue_utc" in a.columns else pd.NA,
    "label":y,
    "aia_input_status":aia_status,
    "sharp_input_status":np.where(sharp_ok,"ok","unavailable"),
    "aia_probability":aia_p,
    "sharp_probability":sharp_p,
    "fusion_probability":fusion_p,
    "aia_seed_std":np.nanstd(aia_seed,axis=1,ddof=1),
    "sharp_seed_std":np.nanstd(sharp_seed,axis=1,ddof=1),
    "fusion_seed_std":np.nanstd(fusion_seed,axis=1,ddof=1),
    "sharp_aia_probability_gap":np.abs(sharp_p-aia_p),
    "fusion_entropy":entropy_binary(fusion_p),
})

sig["aia_conformal_state"]=conformal_state(aia_p,qhat["aia"])
sig["sharp_conformal_state"]=conformal_state(sharp_p,qhat["sharp"])
sig["fusion_conformal_state"]=conformal_state(fusion_p,qhat["sharp_aia"])

# ---------- Baseline ----------
baseline_issue=fusion_ok
baseline_pred=(fusion_p>=alarm_map["sharp_aia"]).astype(int)

# ---------- Candidate policies ----------
def cutoff(metric,pct):
    hit=cutoffs.loc[(cutoffs["metric"]==metric)&np.isclose(cutoffs["percentile"].astype(float),pct)]
    require(len(hit)==1, f"Missing cutoff {metric} q{pct}")
    return float(hit.iloc[0]["cutoff"])

policy_rows=[]
state_assignments={}

def evaluate_policy(policy_id,pct):
    fs=cutoff("fusion_seed_std",pct)
    gap=cutoff("sharp_aia_probability_gap",pct)
    ent=cutoff("fusion_entropy",pct)
    ss=cutoff("sharp_seed_std",pct)

    fusion_single=np.isin(sig["fusion_conformal_state"].to_numpy(),["singleton_no_flare","singleton_flare"])
    sharp_single=np.isin(sig["sharp_conformal_state"].to_numpy(),["singleton_no_flare","singleton_flare"])

    normal=(
        fusion_ok &
        fusion_single &
        (sig["fusion_seed_std"].to_numpy()<=fs) &
        (sig["sharp_aia_probability_gap"].to_numpy()<=gap) &
        (sig["fusion_entropy"].to_numpy()<=ent)
    )
    degraded=(
        (~normal) &
        sharp_ok &
        sharp_single &
        (sig["sharp_seed_std"].to_numpy()<=ss)
    )
    abstain=~(normal|degraded)

    require(np.all((normal.astype(int)+degraded.astype(int)+abstain.astype(int))==1),
            f"States not exhaustive for {policy_id}")

    issued=normal|degraded
    pred=np.full(len(y),-1,dtype=int)
    pred[normal]=(fusion_p[normal]>=alarm_map["sharp_aia"]).astype(int)
    pred[degraded]=(sharp_p[degraded]>=alarm_map["sharp"]).astype(int)

    state=np.where(normal,"NORMAL",np.where(degraded,"DEGRADED","ABSTAIN"))
    state_assignments[policy_id]=state

    issued_y=y[issued]
    issued_pred=pred[issued]
    issued_metrics=binary_metrics(issued_y,issued_pred) if issued.any() else {}

    positives=int((y==1).sum())
    negatives=int((y==0).sum())
    pos_issued=int(((y==1)&issued).sum())
    pos_abstained=int(((y==1)&abstain).sum())
    neg_abstained=int(((y==0)&abstain).sum())

    # Operational false alarms among all requested cases = issued positive predictions on y=0.
    false_alarms=int(((y==0)&issued&(pred==1)).sum())
    detected_flares=int(((y==1)&issued&(pred==1)).sum())
    issued_but_missed=int(((y==1)&issued&(pred==0)).sum())

    row={
        "policy_id":policy_id,
        "cutoff_percentile":pct,
        "requested_cases":len(y),
        "positives":positives,
        "negatives":negatives,
        "normal_cases":int(normal.sum()),
        "degraded_cases":int(degraded.sum()),
        "abstain_cases":int(abstain.sum()),
        "availability_rate":float(issued.mean()),
        "normal_rate":float(normal.mean()),
        "degraded_rate":float(degraded.mean()),
        "abstention_rate":float(abstain.mean()),
        "positive_retention_rate":float(pos_issued/positives) if positives else None,
        "positive_abstention_rate":float(pos_abstained/positives) if positives else None,
        "negative_abstention_rate":float(neg_abstained/negatives) if negatives else None,
        "detected_flares":detected_flares,
        "issued_but_missed_flares":issued_but_missed,
        "abstained_flares":pos_abstained,
        "false_alarms":false_alarms,
        **{f"issued_{k}":v for k,v in issued_metrics.items()}
    }
    return row,normal,degraded,abstain,pred

for policy_id,pct in [("policy_q80",0.80),("policy_q90",0.90),("policy_q95",0.95)]:
    row,normal,degraded,abstain,pred=evaluate_policy(policy_id,pct)
    policy_rows.append(row)
    sig[f"{policy_id}_state"]=state_assignments[policy_id]
    sig[f"{policy_id}_issued_prediction"]=np.where(pred>=0,pred,np.nan)

# Baseline metrics.
bm=binary_metrics(y[baseline_issue],baseline_pred[baseline_issue])
baseline_row={
    "policy_id":"baseline_always_issue_fusion_when_available",
    "cutoff_percentile":np.nan,
    "requested_cases":len(y),
    "positives":int((y==1).sum()),
    "negatives":int((y==0).sum()),
    "normal_cases":int(baseline_issue.sum()),
    "degraded_cases":0,
    "abstain_cases":int((~baseline_issue).sum()),
    "availability_rate":float(baseline_issue.mean()),
    "normal_rate":float(baseline_issue.mean()),
    "degraded_rate":0.0,
    "abstention_rate":float((~baseline_issue).mean()),
    "positive_retention_rate":float(((y==1)&baseline_issue).sum()/(y==1).sum()),
    "positive_abstention_rate":float(((y==1)&(~baseline_issue)).sum()/(y==1).sum()),
    "negative_abstention_rate":float(((y==0)&(~baseline_issue)).sum()/(y==0).sum()),
    "detected_flares":int(((y==1)&baseline_issue&(baseline_pred==1)).sum()),
    "issued_but_missed_flares":int(((y==1)&baseline_issue&(baseline_pred==0)).sum()),
    "abstained_flares":int(((y==1)&(~baseline_issue)).sum()),
    "false_alarms":int(((y==0)&baseline_issue&(baseline_pred==1)).sum()),
    **{f"issued_{k}":v for k,v in bm.items()}
}

results=pd.DataFrame([baseline_row]+policy_rows)

# ---------- Empirical conformal coverage on policy_validation ----------
coverage_rows=[]
for branch,p in [("sharp",sharp_p),("aia",aia_p),("sharp_aia",fusion_p)]:
    finite,contains,size=conformal_contains_true(y,p,qhat[branch])
    coverage_rows.append({
        "branch":branch,
        "alpha":PRIMARY_ALPHA,
        "nominal_coverage":1-PRIMARY_ALPHA,
        "available_cases":int(finite.sum()),
        "empirical_coverage":float(contains[finite].mean()) if finite.any() else None,
        "empty_set_rate":float((size[finite]==0).mean()) if finite.any() else None,
        "singleton_rate":float((size[finite]==1).mean()) if finite.any() else None,
        "doubleton_rate":float((size[finite]==2).mean()) if finite.any() else None,
        "positive_coverage":float(contains[(y==1)&finite].mean()) if ((y==1)&finite).any() else None,
        "negative_coverage":float(contains[(y==0)&finite].mean()) if ((y==0)&finite).any() else None,
    })
coverage=pd.DataFrame(coverage_rows)

# ---------- Branch probabilistic + threshold metrics on full policy-validation ----------
branch_rows=[]
for branch,p,available in [
    ("sharp",sharp_p,sharp_ok),
    ("aia",aia_p,aia_ok),
    ("sharp_aia",fusion_p,fusion_ok),
]:
    pred=(p>=alarm_map[branch]).astype(int)
    m=binary_metrics(y[available],pred[available])
    branch_rows.append({
        "branch":branch,
        "available_cases":int(available.sum()),
        "availability_rate":float(available.mean()),
        "brier":brier(y,p),
        "log_loss":logloss(y,p),
        **m,
    })
branch_metrics=pd.DataFrame(branch_rows)

# ---------- Primary verdict ----------
primary=results.loc[results["policy_id"]=="policy_q90"].iloc[0].to_dict()
baseline=results.loc[results["policy_id"]=="baseline_always_issue_fusion_when_available"].iloc[0].to_dict()

# This is descriptive, not an automatic scientific-acceptance gate.
comparison={
    "primary_policy_id":"policy_q90",
    "baseline_policy_id":"baseline_always_issue_fusion_when_available",
    "availability_delta":float(primary["availability_rate"]-baseline["availability_rate"]),
    "positive_retention_delta":float(primary["positive_retention_rate"]-baseline["positive_retention_rate"]),
    "issued_tss_delta":float(primary["issued_tss"]-baseline["issued_tss"]),
    "false_alarm_delta":int(primary["false_alarms"]-baseline["false_alarms"]),
    "abstained_flare_delta":int(primary["abstained_flares"]-baseline["abstained_flares"]),
    "warning":(
        "Issued-only skill must not be interpreted without availability and flare-retention. "
        "Abstained positives are reported explicitly."
    )
}

results_path=OUT/"policy_results.csv"
coverage_path=OUT/"conformal_policy_validation_coverage.csv"
branch_path=OUT/"branch_metrics.csv"
cases_path=OUT/"policy_validation_cases.csv.gz"
summary_path=OUT/"summary.json"

results.to_csv(results_path,index=False)
coverage.to_csv(coverage_path,index=False)
branch_metrics.to_csv(branch_path,index=False)
sig.to_csv(cases_path,index=False,compression={"method":"gzip","mtime":0})

summary={
    "status":"POLICY_VALIDATION_EVALUATED_WITH_FROZEN_RULES",
    "scientific_acceptance":False,
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "role":ROLE,
    "rows":len(y),
    "positives":int((y==1).sum()),
    "primary_policy_id":"policy_q90",
    "primary_was_predeclared":True,
    "policy_reselection_performed":False,
    "later_roles_read":False,
    "alarm_thresholds":alarm_map,
    "primary_conformal_alpha":PRIMARY_ALPHA,
    "comparison_primary_vs_baseline":comparison,
    "source_sha256":{
        "master_predictions":MASTER_SHA,
        "sharp_predictions":SHARP_SHA,
        "conformal_protocol":sha256(CONF_PROTOCOL),
        "conformal_thresholds":sha256(CONF_THRESHOLDS),
        "policy_protocol":sha256(POLICY_PROTOCOL),
        "policy_family":sha256(POLICY_FAMILY),
        "candidate_cutoffs":sha256(CUTOFFS),
        "alarm_protocol":sha256(ALARM_PROTOCOL),
        "alarm_thresholds":sha256(ALARM_THRESHOLDS),
        "primary_policy_declaration":sha256(PRIMARY_DECL),
    },
    "output_sha256":{
        results_path.name:sha256(results_path),
        coverage_path.name:sha256(coverage_path),
        branch_path.name:sha256(branch_path),
        cases_path.name:sha256(cases_path),
    },
    "note":"First outcome evaluation of frozen operational policies; Cycle 25 and supplementary 2026 remain unopened here."
}
summary_path.write_text(json.dumps(summary,indent=2)+"\n")

print("===== POLICY VALIDATION COMPLETE =====")
print("rows:",len(y))
print("positives:",int((y==1).sum()))
print("primary policy: policy_q90 (predeclared)")
print("policy reselection performed: False")
print("later roles read: False")

print("\nBranch metrics:")
print(branch_metrics.to_string(index=False))

print("\nEmpirical conformal coverage:")
print(coverage.to_string(index=False))

print("\nOperational policy results:")
print(results.to_string(index=False))

print("\nPrimary q90 vs baseline:")
print(json.dumps(comparison,indent=2))

print("\nSaved:",summary_path)
print("Saved:",results_path)
print("Saved:",coverage_path)
print("Saved:",branch_path)
print("Saved:",cases_path)
